# 01 · 数据获取与清洗 —— 先"体检"，再"动刀"

**项目**：英国在线零售商经营诊断分析（UCI Online Retail 公开数据集）

**业务背景**：你是一家英国在线礼品零售商的数据分析师。这家公司主要卖礼品杂货，客户八成是英国的批发企业（B2B），也有一小部分海外零售客户。老板最近想知道：**生意到底在变好还是变坏？钱是谁贡献的？哪些环节在漏水？**

**数据集**：[UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)，2010-12-01 ~ 2011-12-09 的真实交易流水，54 万余行。

| 字段 | 含义 | 备注 |
| --- | --- | --- |
| InvoiceNo | 发票号（一次交易） | 6 位数字；**C 开头 = 取消/退货单** |
| StockCode | 商品编码 | 混有运费(POST)、银行手续费(BANK CHARGES)等非商品行 |
| Description | 商品描述 | |
| Quantity | 数量 | **负数 = 退货** |
| InvoiceDate | 开票时间 | 精确到分钟 |
| UnitPrice | 单价（英镑 £） | 有 0 元与负数 |
| CustomerID | 客户编号 | **约 25% 缺失**（多为无主批发展客） |
| Country | 国家 | 约 38 个 |

**本 Notebook 做两件事**：① 对原始数据做全面"体检"，摸清脏数据在哪；② 逐条清洗，每一步给出**业务理由**，最终产出干净的销售明细表和退货明细表，供后续分析使用。

## 0. 环境准备

> **拆解**：统一在这里完成三件事——导入库、解决 matplotlib 的中文乱码、统一 pandas 的显示格式。每个 Notebook 开头都有这一段，保证任何一个 Notebook 单独打开都能跑通。

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# 中文显示（Windows 自带微软雅黑）；负号正常显示
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 千分位、两位小数，金额一眼可读
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 50)

print('pandas 版本:', pd.__version__)

pandas 版本: 2.3.3


## 1. 数据加载与"体检"

> **拆解**：拿到数据的正确姿势**不是马上动手改**，而是先量化"脏"在哪里。体检四件套：**缺失、重复、异常值、口径问题**（类型不对、时间越界）。体检结果决定清洗方案。

In [2]:
# 54 万行 Excel，读取约 1~2 分钟；后面所有 Notebook 都改读清洗后的 CSV，秒级加载
raw = pd.read_excel('../data/raw/Online Retail.xlsx')
print(f'原始数据规模：{raw.shape[0]:,} 行 × {raw.shape[1]} 列')
raw.head()

原始数据规模：541,909 行 × 8 列


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


In [3]:
# 体检①：缺失情况 —— 哪个字段缺、缺多少
quality = pd.DataFrame({
    '缺失数量': raw.isna().sum(),
    '缺失占比(%)': (raw.isna().mean() * 100).round(2),
})
quality[quality['缺失数量'] > 0]

,缺失数量,缺失占比(%)
Description,1454,0.27
CustomerID,135080,24.93


In [4]:
# 体检②：重复行 / 时间范围 / 数值分布
print('完全重复行数:', raw.duplicated().sum())
print('时间范围:', raw['InvoiceDate'].min(), '→', raw['InvoiceDate'].max())
raw[['Quantity', 'UnitPrice']].describe()

完全重复行数: 5268
时间范围: 2010-12-01 08:26:00 → 2011-12-09 12:50:00


,Quantity,UnitPrice
count,"541,909.00","541,909.00"
mean,9.55,4.61
std,218.08,96.76
min,"-80,995.00","-11,062.06"
25%,1.00,1.25
50%,3.00,2.08
75%,10.00,4.13
max,"80,995.00","38,970.00"


In [5]:
# 体检③：异常值解剖 —— 负数量、0 元单价、取消单
cancel_mask = raw['InvoiceNo'].astype(str).str.startswith('C')
print(f'数量≤0 的行数:        {(raw["Quantity"] <= 0).sum():,}')
print(f'单价≤0 的行数:        {(raw["UnitPrice"] <= 0).sum():,}')
print(f'发票号 C 开头(取消单): {cancel_mask.sum():,}')
print(f'最后一天(2011-12-09)行数: {(raw["InvoiceDate"] >= "2011-12-09").sum():,}')
raw.loc[cancel_mask, ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate']].head()

数量≤0 的行数:        10,624
单价≤0 的行数:        2,517
发票号 C 开头(取消单): 9,288
最后一天(2011-12-09)行数: 1,632


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate
141,C536379,D,Discount,-1,2010-12-01 09:41:00
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00


In [6]:
# 体检④：混进发票里的"非商品行" —— 运费、手续费、赠品、测试单
codes = raw['StockCode'].astype(str)
no_digit = raw.loc[~codes.str.contains(r'\d'), ['StockCode', 'Description']].drop_duplicates()
print('不含数字的特殊编码：')
no_digit

不含数字的特殊编码：


,StockCode,Description
45,POST,POSTAGE
141,D,Discount
1814,DOT,DOTCOM POSTAGE
2239,M,Manual
4406,BANK CHARGES,Bank Charges
14436,S,SAMPLES
14514,AMAZONFEE,AMAZON FEE
40383,m,Manual
52262,POST,NaN
84016,DCGSSBOY,BOYS PARTY BAG


### 体检结论

| 问题 | 规模 | 对分析的影响 | 处理决策 |
| --- | --- | --- | --- |
| CustomerID 缺失 | 约 25% | 无法归属到人，RFM/留存做不了 | **销售表剔除**（多为无主批发展客），占比单独披露 |
| Quantity ≤ 0（C 单） | 约 1 万行 | 混入销售会算出"负 GMV" | **拆出独立退货表**，退货本身是重要业务问题 |
| UnitPrice ≤ 0 | 少量 | 0 元行多为赠品/坏账调整，会扭曲客单价 | 销售表剔除 |
| 非商品行（POST/M/DOT…） | 少量 | 运费、手续费不是"卖了什么" | 销售表剔除 |
| 完全重复行 | 约 5 千行 | 重复计数虚增 GMV | 去重 |
| 最后一天 2011-12-09 | 全天数据只采集到中午 | 保留会让 12 月"暴跌"，**误判业务** | 剔除该天，分析期止于 2011-12-08 |

> **面试考点**：为什么必须剔掉 2011-12-09？——数据在当天上午停止采集，12 月只剩大半天。如果直接画月度趋势图，12 月 GMV 会"断崖式下跌"，业务方会误以为经营出问题。**先对齐统计口径，再看趋势**，这是数据分析师的职业本能。

## 2. 逐条清洗（每步附业务理由）

> **拆解**：清洗顺序有讲究——**先做"拆分"再做"过滤"**：先把退货单从主表里拆出来（它们本身是有价值的分析对象），再对销售表逐条过滤，每一步都打印前后行数，形成可回溯的清洗日志。

In [7]:
df = raw.copy()
steps = []   # 清洗日志：记录每一步前后的行数

def apply_step(name, mask_keep, note=''):
    """对 df 应用布尔掩码（True=保留），并记录清洗日志"""
    before = len(df)
    kept = int(mask_keep.sum())
    steps.append({'步骤': name, '清洗前行数': f'{before:,}',
                  '剔除行数': f'{before - kept:,}', '剩余行数': f'{kept:,}',
                  '说明': note})
    return df[mask_keep].copy()

In [8]:
# 2.1 时间类型转换 + 剔除不完整的最后一天
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = apply_step('剔除不完整末日 2011-12-09',
                df['InvoiceDate'] < '2011-12-09',
                '当天数据只采集到上午，保留会误判 12 月暴跌')
print('分析期:', df['InvoiceDate'].min(), '→', df['InvoiceDate'].max())

分析期: 2010-12-01 08:26:00 → 2011-12-08 20:01:00


In [9]:
# 2.2 拆分：退货/取消表 vs 正常销售表（拆分不等于丢弃，退货是第 7 个业务问题的主角）
cancel_mask = df['InvoiceNo'].astype(str).str.startswith('C') | (df['Quantity'] < 0)
returns = df[cancel_mask].copy()
df = df[~cancel_mask].copy()
steps.append({'步骤': '拆分退货/取消单', '清洗前行数': f'{len(df) + len(returns):,}',
              '剔除行数': f'{len(returns):,}（转入退货表）',
              '剩余行数': f'{len(df):,}', '说明': 'C 开头发票或负数量，单独成表分析'})
print(f'退货/取消 {len(returns):,} 行 | 正常销售 {len(df):,} 行')

退货/取消 10,617 行 | 正常销售 529,660 行


In [10]:
# 2.3 剔除缺失 CustomerID 的销售行（先量化它"值多少钱"，再动手）
n_missing = df['CustomerID'].isna().sum()
money_missing = df.loc[df['CustomerID'].isna(), 'Quantity'].mul(df['UnitPrice']).sum()
print(f'缺失客户ID：{n_missing:,} 行（{n_missing / len(df):.1%}），对应销售额 £{money_missing:,.0f}')
df = apply_step('剔除缺失 CustomerID', df['CustomerID'].notna(),
                '无法归属到人，RFM/留存做不了；多为无主批发展客')

缺失客户ID：132,353 行（25.0%），对应销售额 £1,716,581


In [11]:
# 2.4 剔除非商品行 + 0 元/负价行
special_codes = {'POST', 'DOT', 'M', 'D', 'CRUK', 'C2', 'PADS', 'S',
                 'BANK CHARGES', 'ADJUST', 'AMAZONFEE', 'GIFT'}
non_product_mask = df['StockCode'].astype(str).str.upper().isin(special_codes)
print(f'非商品行：{non_product_mask.sum():,} 行')
df = apply_step('剔除非商品编码', ~non_product_mask, '运费/手续费/赠品/测试行，不属于"卖了什么"')
df = apply_step('剔除单价≤0', df['UnitPrice'] > 0, '0 元多为赠品/内部调整，会扭曲客单价')

非商品行：1,549 行


In [12]:
# 2.5 去重 + 派生分析字段
before = len(df)
df = df.drop_duplicates()
steps.append({'步骤': '完全重复行去重', '清洗前行数': f'{before:,}',
              '剔除行数': f'{before - len(df):,}', '剩余行数': f'{len(df):,}',
              '说明': '同一行重复记录会虚增 GMV'})

# 派生字段：Revenue 是全项目的核心度量，InvoiceMonth 是月度分析的主口径
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['CustomerID'] = df['CustomerID'].astype(int)          # 去掉缺失后可安全转整型
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
print('最终销售明细：', f'{len(df):,} 行')

最终销售明细： 390,545 行


## 3. 清洗结果报告

In [13]:
log = pd.DataFrame(steps)
display(log)
print()
print(f'原始 {len(raw):,} 行 → 销售明细 {len(df):,} 行（保留率 {len(df) / len(raw):.1%}）')
print(f'退货/取消明细 {len(returns):,} 行，单独成表')
print(f'客户数：{df["CustomerID"].nunique():,} | 商品数：{df["StockCode"].nunique():,} | 国家数：{df["Country"].nunique()}')

,步骤,清洗前行数,剔除行数,剩余行数,说明
0,剔除不完整末日 2011-12-09,"541,909","1,632","540,277",当天数据只采集到上午，保留会误判 12 月暴跌
1,拆分退货/取消单,"540,277","10,617（转入退货表）","529,660",C 开头发票或负数量，单独成表分析
2,剔除缺失 CustomerID,"529,660","132,353","397,307",无法归属到人，RFM/留存做不了；多为无主批发展客
3,剔除非商品编码,"397,307","1,549","395,758","运费/手续费/赠品/测试行，不属于""卖了什么"""
4,剔除单价≤0,"395,758",33,"395,725",0 元多为赠品/内部调整，会扭曲客单价
5,完全重复行去重,"395,725","5,180","390,545",同一行重复记录会虚增 GMV



原始 541,909 行 → 销售明细 390,545 行（保留率 72.1%）
退货/取消明细 10,617 行，单独成表
客户数：4,333 | 商品数：3,658 | 国家数：37


In [14]:
# 落盘：后续 Notebook 只读这两张干净表，不再碰原始 Excel
df.to_csv('../data/cleaned/sales_clean.csv', index=False)
returns.to_csv('../data/cleaned/returns_clean.csv', index=False)
print('已保存: data/cleaned/sales_clean.csv 与 data/cleaned/returns_clean.csv')
df.head()

已保存: data/cleaned/sales_clean.csv 与 data/cleaned/returns_clean.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,InvoiceMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12


### 清洗小结（面试怎么讲）

1. **先体检后清洗**：缺失 → 重复 → 异常 → 口径，四查给出量化体检表，而不是拍脑袋删数据；
2. **每一刀都有业务理由**：剔除缺失客户 ID 是因为"客户分析无法归属"，剔除末日是因为"口径不完整"，剔除 0 元行是因为"会扭曲客单价"；
3. **拆分优于丢弃**：退货单没有删掉，而是拆成独立表，成为"退货诊断"专题的数据源；
4. **全程留痕**：每步打印前后行数，任何同行质疑都能回溯。